# 🤖 Capa 3: Agente LLM con OpenAI

Este notebook conecta el chatbot con OpenAI y define:
- System prompt del agente
- Tools/funciones disponibles para el LLM
- Manejo de contexto conversacional
- Lógica de function calling

In [21]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv

# Cargar variables de entorno
load_dotenv('../.env')

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
print('✅ Conexión con OpenAI establecida')

✅ Conexión con OpenAI establecida


## 1. System Prompt

Define la personalidad y reglas del chatbot.

In [22]:
SYSTEM_PROMPT = """
Eres EcoBot, el asistente virtual del supermercado EcoMarket. Eres amable, eficiente y servicial.

CAPACIDADES:
1. Consultar disponibilidad de productos (por nombre o categoría)
2. Informar sobre el estado de pedidos (el cliente se identifica con su email)
3. Mostrar promociones vigentes y recomendar según lo que consulte el cliente
4. Gestionar devoluciones (registrar nuevas, consultar estado)
5. Derivar a un agente humano cuando no puedas resolver algo

REGLAS:
- Siempre responde en español
- Sé conciso pero amable
- Si un producto tiene promoción activa, menciónala al mostrar info del producto
- Para pedidos y devoluciones, siempre pide el email del cliente si no lo tienes
- Si no puedes resolver algo, ofrece derivar a un agente humano
- Para derivar a humano, necesitas: email, descripción del problema, y opcionalmente número de pedido
- No inventes información. Si no encuentras datos, dilo claramente
- Usa emojis moderadamente para ser más amigable
"""

print('System prompt definido ✅')

System prompt definido ✅


## 2. Definición de Tools (Function Calling)

Definimos las herramientas que el LLM puede invocar.

In [23]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "buscar_producto",
            "description": "Busca productos por nombre en el catálogo. Devuelve nombre, precio, stock y categoría.",
            "parameters": {
                "type": "object",
                "properties": {
                    "nombre": {
                        "type": "string",
                        "description": "Nombre o parte del nombre del producto a buscar"
                    }
                },
                "required": ["nombre"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "buscar_por_categoria",
            "description": "Lista todos los productos de una categoría específica. Categorías: Frutas y Verduras, Lácteos, Carnes, Panadería, Bebidas, Limpieza y Hogar, Despensa",
            "parameters": {
                "type": "object",
                "properties": {
                    "categoria": {
                        "type": "string",
                        "description": "Categoría de productos"
                    }
                },
                "required": ["categoria"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_pedidos",
            "description": "Consulta el estado de los pedidos de un cliente. Requiere el email del cliente.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {
                        "type": "string",
                        "description": "Email del cliente"
                    }
                },
                "required": ["email"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "obtener_detalle_pedido",
            "description": "Obtiene el detalle completo de un pedido específico (productos, cantidades, precios).",
            "parameters": {
                "type": "object",
                "properties": {
                    "pedido_id": {
                        "type": "integer",
                        "description": "ID del pedido"
                    }
                },
                "required": ["pedido_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "obtener_promociones",
            "description": "Muestra todas las promociones vigentes con descuentos y precios.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "registrar_devolucion",
            "description": "Registra una solicitud de devolución. Requiere pedido_id, email y motivo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "pedido_id": {
                        "type": "integer",
                        "description": "ID del pedido a devolver"
                    },
                    "email": {
                        "type": "string",
                        "description": "Email del cliente"
                    },
                    "motivo": {
                        "type": "string",
                        "description": "Razón de la devolución"
                    }
                },
                "required": ["pedido_id", "email", "motivo"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "consultar_devoluciones",
            "description": "Consulta el estado de las devoluciones de un cliente.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {
                        "type": "string",
                        "description": "Email del cliente"
                    }
                },
                "required": ["email"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "crear_ticket_soporte",
            "description": "Crea un ticket para derivar el caso a un agente humano. Usar cuando el bot no puede resolver el problema.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {
                        "type": "string",
                        "description": "Email del cliente"
                    },
                    "asunto": {
                        "type": "string",
                        "description": "Asunto breve del problema"
                    },
                    "descripcion": {
                        "type": "string",
                        "description": "Descripción detallada del problema"
                    },
                    "pedido_id": {
                        "type": "integer",
                        "description": "ID del pedido relacionado (opcional)"
                    }
                },
                "required": ["email", "asunto", "descripcion"]
            }
        }
    }
]

print(f'✅ {len(TOOLS)} herramientas definidas para el agente')

✅ 8 herramientas definidas para el agente


## 3. Mapeo de funciones

Conectamos los nombres de las tools con las funciones reales.

In [24]:
# Importar funciones de consulta
import sqlite3
import os
from datetime import datetime

DB_PATH = os.path.join(os.getcwd(), '..', 'data', 'ecomarket.db')
if not os.path.exists(DB_PATH):
    DB_PATH = os.path.join(os.getcwd(), 'data', 'ecomarket.db')

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def buscar_producto(nombre: str) -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id, nombre, categoria, precio, stock, descripcion, unidad FROM productos WHERE LOWER(nombre) LIKE LOWER(?)', (f'%{nombre}%',))
    resultados = [dict(row) for row in cursor.fetchall()]
    hoy = datetime.now().strftime('%Y-%m-%d')
    for prod in resultados:
        cursor.execute('SELECT descripcion, descuento_porcentaje FROM promociones WHERE producto_id = ? AND activa = 1 AND fecha_inicio <= ? AND fecha_fin >= ?', (prod['id'], hoy, hoy))
        promo = cursor.fetchone()
        if promo:
            prod['promocion'] = dict(promo)
    conn.close()
    return resultados

def buscar_por_categoria(categoria: str) -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id, nombre, categoria, precio, stock, unidad FROM productos WHERE LOWER(categoria) LIKE LOWER(?) ORDER BY nombre', (f'%{categoria}%',))
    resultados = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return resultados

def consultar_pedidos(email: str) -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id, fecha_pedido, estado, total, direccion_envio, fecha_entrega_estimada FROM pedidos WHERE LOWER(email_cliente) = LOWER(?) ORDER BY fecha_pedido DESC', (email,))
    resultados = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return resultados

def obtener_detalle_pedido(pedido_id: int) -> dict:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT * FROM pedidos WHERE id = ?', (pedido_id,))
    pedido = cursor.fetchone()
    if not pedido:
        conn.close()
        return {'error': 'Pedido no encontrado'}
    cursor.execute('SELECT p.nombre, dp.cantidad, dp.precio_unitario, (dp.cantidad * dp.precio_unitario) as subtotal FROM detalle_pedido dp JOIN productos p ON p.id = dp.producto_id WHERE dp.pedido_id = ?', (pedido_id,))
    items = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return {'pedido': dict(pedido), 'items': items}

def obtener_promociones() -> list:
    conn = get_connection()
    cursor = conn.cursor()
    hoy = datetime.now().strftime('%Y-%m-%d')
    cursor.execute('SELECT pr.id, p.nombre as producto, pr.descripcion, pr.descuento_porcentaje, pr.fecha_fin, p.precio as precio_original, ROUND(p.precio * (1 - pr.descuento_porcentaje/100), 2) as precio_con_descuento FROM promociones pr JOIN productos p ON p.id = pr.producto_id WHERE pr.activa = 1 AND pr.fecha_inicio <= ? AND pr.fecha_fin >= ? ORDER BY pr.descuento_porcentaje DESC', (hoy, hoy))
    resultados = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return resultados

def registrar_devolucion(pedido_id: int, email: str, motivo: str) -> dict:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id, estado FROM pedidos WHERE id = ? AND LOWER(email_cliente) = LOWER(?)', (pedido_id, email))
    pedido = cursor.fetchone()
    if not pedido:
        conn.close()
        return {'error': 'Pedido no encontrado o no pertenece a este email'}
    if dict(pedido)['estado'] not in ['entregado', 'enviado']:
        conn.close()
        return {'error': f'No se puede devolver un pedido con estado: {dict(pedido)["estado"]}'}
    fecha = datetime.now().strftime('%Y-%m-%d')
    cursor.execute('INSERT INTO devoluciones (pedido_id, email_cliente, motivo, estado, fecha_solicitud) VALUES (?, ?, ?, "pendiente", ?)', (pedido_id, email, motivo, fecha))
    devolucion_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return {'mensaje': 'Devolucion registrada', 'devolucion_id': devolucion_id, 'estado': 'pendiente'}

def consultar_devoluciones(email: str) -> list:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT id, pedido_id, motivo, estado, fecha_solicitud, comentarios FROM devoluciones WHERE LOWER(email_cliente) = LOWER(?) ORDER BY fecha_solicitud DESC', (email,))
    resultados = [dict(row) for row in cursor.fetchall()]
    conn.close()
    return resultados

def crear_ticket_soporte(email: str, asunto: str, descripcion: str, pedido_id: int = None) -> dict:
    conn = get_connection()
    cursor = conn.cursor()
    fecha = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    cursor.execute('INSERT INTO tickets_soporte (email_cliente, asunto, descripcion, estado, fecha_creacion, pedido_id) VALUES (?, ?, ?, "abierto", ?, ?)', (email, asunto, descripcion, fecha, pedido_id))
    ticket_id = cursor.lastrowid
    conn.commit()
    conn.close()
    return {'mensaje': 'Ticket creado. Un agente se pondra en contacto pronto.', 'ticket_id': ticket_id}

print(f'Funciones cargadas. DB: {DB_PATH}')
print(f'DB existe: {os.path.exists(DB_PATH)}')

Funciones cargadas. DB: /Users/dannasarmientomartinez/Documents/Documentos - MacBook Air de Danna/Master Business Analytics/chatbot_ecomarket/notebooks/../data/ecomarket.db
DB existe: True


In [25]:
# Mapeo nombre_funcion -> función real
FUNCIONES_DISPONIBLES = {
    'buscar_producto': buscar_producto,
    'buscar_por_categoria': buscar_por_categoria,
    'consultar_pedidos': consultar_pedidos,
    'obtener_detalle_pedido': obtener_detalle_pedido,
    'obtener_promociones': obtener_promociones,
    'registrar_devolucion': registrar_devolucion,
    'consultar_devoluciones': consultar_devoluciones,
    'crear_ticket_soporte': crear_ticket_soporte,
}

def ejecutar_funcion(nombre_funcion: str, argumentos: dict) -> str:
    """Ejecuta una función por nombre y devuelve el resultado como JSON string."""
    if nombre_funcion not in FUNCIONES_DISPONIBLES:
        return json.dumps({'error': f'Función {nombre_funcion} no encontrada'})
    
    funcion = FUNCIONES_DISPONIBLES[nombre_funcion]
    # Manejar caso donde argumentos es None o vacío
    if not argumentos:
        argumentos = {}
    resultado = funcion(**argumentos)
    return json.dumps(resultado, ensure_ascii=False, default=str)

print('✅ Mapeo de funciones configurado')

✅ Mapeo de funciones configurado


## 4. Motor de conversación

Función principal que gestiona el ciclo conversacional con function calling.

In [26]:
MODEL = 'gpt-4o-mini'  # Modelo de OpenAI

def chat(mensaje_usuario: str, historial: list) -> tuple[str, list]:
    """
    Procesa un mensaje del usuario y devuelve la respuesta del agente.
    
    Args:
        mensaje_usuario: Texto del usuario
        historial: Lista de mensajes previos (contexto)
    
    Returns:
        (respuesta_texto, historial_actualizado)
    """
    # Agregar mensaje del usuario al historial
    historial.append({'role': 'user', 'content': mensaje_usuario})
    
    # Construir mensajes con system prompt
    mensajes = [{'role': 'system', 'content': SYSTEM_PROMPT}] + historial
    
    # Llamar a OpenAI
    response = client.chat.completions.create(
        model=MODEL,
        messages=mensajes,
        tools=TOOLS,
        tool_choice='auto',
        max_tokens=1024,
        temperature=0.7
    )
    
    message = response.choices[0].message
    
    # Si el modelo quiere llamar funciones
    while message.tool_calls:
        # Agregar respuesta del asistente al historial
        historial.append({
            'role': 'assistant',
            'content': message.content,
            'tool_calls': [
                {
                    'id': tc.id,
                    'type': 'function',
                    'function': {
                        'name': tc.function.name,
                        'arguments': tc.function.arguments
                    }
                } for tc in message.tool_calls
            ]
        })
        
        # Ejecutar cada función solicitada
        for tool_call in message.tool_calls:
            nombre_fn = tool_call.function.name
            args = json.loads(tool_call.function.arguments) if tool_call.function.arguments else {}
            
            print(f'  🔧 Ejecutando: {nombre_fn}({args})')
            resultado = ejecutar_funcion(nombre_fn, args)
            
            # Agregar resultado al historial
            historial.append({
                'role': 'tool',
                'tool_call_id': tool_call.id,
                'content': resultado
            })
        
        # Volver a llamar al modelo con los resultados
        mensajes = [{'role': 'system', 'content': SYSTEM_PROMPT}] + historial
        response = client.chat.completions.create(
            model=MODEL,
            messages=mensajes,
            tools=TOOLS,
            tool_choice='auto',
            max_tokens=1024,
            temperature=0.7
        )
        message = response.choices[0].message
    
    # Respuesta final (sin tool calls)
    respuesta = message.content
    historial.append({'role': 'assistant', 'content': respuesta})
    
    return respuesta, historial

print('✅ Motor de conversación listo')

✅ Motor de conversación listo


## 5. Prueba del agente

Probamos el chatbot con diferentes consultas.

In [27]:
# Iniciar conversación
historial = []

# Prueba 1: Buscar producto
print('👤 Usuario: ¿Tienen leche disponible?')
respuesta, historial = chat('¿Tienen leche disponible?', historial)
print(f'🤖 EcoBot: {respuesta}')
print('---')

👤 Usuario: ¿Tienen leche disponible?
  🔧 Ejecutando: buscar_producto({'nombre': 'leche'})


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01krh939mdegjtfyyn65jhcnzz` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 2446. Please try again in 35m13.344s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# Prueba 2: Consultar promociones
print('👤 Usuario: ¿Qué promociones tienen hoy?')
respuesta, historial = chat('¿Qué promociones tienen hoy?', historial)
print(f'🤖 EcoBot: {respuesta}')
print('---')

👤 Usuario: ¿Qué promociones tienen hoy?
  🔧 Ejecutando: obtener_promociones(None)
🤖 EcoBot: Tenemos varias promociones disponibles hoy:

1. 2x1 en Manzanas Rojas
2. Pan Integral a mitad de precio los martes
3. Agua Mineral: 3x2
4. 25% descuento en Chocolate Negro
5. Cerveza Artesanal: 4x3
6. 20% descuento en Leche Entera
7. 15% descuento en Pechuga de Pollo
8. 10% descuento en Aceite de Oliva

¿Te gustaría saber más sobre alguna de estas ofertas? 🛍️
---


In [ ]:
# Prueba 3: Consultar pedido
print('👤 Usuario: Quiero ver mis pedidos, mi email es maria.garcia@email.com')
respuesta, historial = chat('Quiero ver mis pedidos, mi email es maria.garcia@email.com', historial)
print(f'🤖 EcoBot: {respuesta}')
print('---')

👤 Usuario: Quiero ver mis pedidos, mi email es maria.garcia@email.com
  🔧 Ejecutando: consultar_pedidos({'email': 'maria.garcia@email.com'})
🤖 EcoBot: ¡Hola María! He encontrado dos pedidos asociados a tu email:

1. Pedido #21 del 17 de junio de 2026, con un total de $18.00, en estado "confirmado" y con una fecha de entrega estimada para el 19 de junio de 2026.
2. Pedido #15 del 23 de mayo de 2026, con un total de $57.12, en estado "enviado" y con una fecha de entrega estimada para el 19 de junio de 2026.

¿Necesitas más información sobre alguno de estos pedidos o hay algo más en lo que pueda ayudarte? 📦
---


In [ ]:
# Prueba 4: Escalar a humano
print('👤 Usuario: Tengo un problema con un cobro duplicado en mi tarjeta')
respuesta, historial = chat('Tengo un problema con un cobro duplicado en mi tarjeta, mi email es maria.garcia@email.com', historial)
print(f'🤖 EcoBot: {respuesta}')
print('---')

👤 Usuario: Tengo un problema con un cobro duplicado en mi tarjeta
  🔧 Ejecutando: crear_ticket_soporte({'asunto': 'Cobro duplicado en tarjeta', 'descripcion': 'Estimados, tengo un problema con un cobro duplicado en mi tarjeta. Necesito ayuda para resolver este asunto lo antes posible. Agradezco su atención.', 'email': 'maria.garcia@email.com'})
🤖 EcoBot: ¡Lo siento mucho, María! Me parece que este es un tema un poco complicado y prefiero que un agente humano se encargue de ayudarte a resolver el problema del cobro duplicado en tu tarjeta. Acabo de crear un ticket de soporte para que puedas obtener ayuda lo antes posible. Un agente se pondrá en contacto contigo pronto para ayudarte a resolver este asunto. ¡Gracias por tu paciencia y espero que se resuelva pronto! 🤗
---
